In [1]:
%reload_ext autoreload
%autoreload 2
%cd /home/hazzu/Code/thesis/

/home/hazzu/Code/thesis


In [2]:
import random, os
import time
import pandas as pd


def create_verification_pairs(dataset_dir):
    random.seed(time.time())

    genuine_pairs = []
    imposter_pairs = []
    identities = [
        d
        for d in os.listdir(dataset_dir)
        if os.path.isdir(os.path.join(dataset_dir, d))
    ]

    if not identities:
        print(f"Không tìm thấy thư mục danh tính nào trong: {dataset_dir}")
        return [], []

    print(f"Tìm thấy {len(identities)} danh tính.")

    print("Đang tạo genuine pairs...")
    for identity_folder in identities:
        identity_path = os.path.join(dataset_dir, identity_folder)
        images_in_identity = [
            os.path.join(identity_path, img)
            for img in os.listdir(identity_path)
            if img.lower().endswith((".png", ".jpg", ".jpeg"))
        ]

        if len(images_in_identity) < 2:
            print(f"Danh tính {identity_folder} có ít hơn 2 ảnh, bỏ qua genuine pairs.")
            assert len(images_in_identity) >= 2

        if len(images_in_identity) == 2:
            pairs = set()
            pairs.add((images_in_identity[0], images_in_identity[1]))
            genuine_pairs.extend(pairs)
            continue

        shuffle_images_in_identity = images_in_identity.copy()
        while True:
            random.shuffle(shuffle_images_in_identity)

            pairs = set()
            is_unique = True
            for image1, image2 in zip(images_in_identity, shuffle_images_in_identity):
                if image1 == image2:
                    is_unique = False
                    break

                if (image1, image2) in pairs or (image2, image1) in pairs:
                    is_unique = False
                    break

                pairs.add((image1, image2))

            if is_unique:
                break

        genuine_pairs.extend(pairs)
    print(f"Đã tạo {len(genuine_pairs)} genuine pairs.")

    print("Đang tạo imposter pairs...")
    all_images_paths_with_identity = []
    for identity_folder in identities:
        identity_path = os.path.join(dataset_dir, identity_folder)
        for img_file in os.listdir(identity_path):
            if img_file.lower().endswith((".png", ".jpg", ".jpeg")):
                all_images_paths_with_identity.append(
                    (os.path.join(identity_path, img_file), identity_folder)
                )

    if not all_images_paths_with_identity or len(identities) < 2:
        print("Không đủ ảnh hoặc danh tính để tạo imposter pairs.")
        return genuine_pairs, imposter_pairs

    for image1, identity1 in all_images_paths_with_identity:
        identity2 = None
        while True:
            image2, identity2 = random.choice(all_images_paths_with_identity)
            if identity1 != identity2:
                break

        imposter_pairs.append((image1, image2))
    print(f"Đã tạo {len(imposter_pairs)} imposter pairs.")

    return genuine_pairs, imposter_pairs


def save_pairs_to_csv(pairs, file_path):
    df = pd.DataFrame(
        {
            "image1_path": [pair[0] for pair in pairs],
            "image2_path": [pair[1] for pair in pairs],
        }
    )
    df.to_csv(file_path, index=False)
    print(f"Đã lưu {len(pairs)} pairs vào {file_path}")

In [3]:
genuine_pairs_train, imposter_pairs_train = create_verification_pairs(
    "datasets/ver1/selfies_2"
)
genuine_pairs_test, imposter_pairs_test = create_verification_pairs(
    "datasets/ver1/selfies_1"
)

Tìm thấy 70 danh tính.
Đang tạo genuine pairs...
Đã tạo 239 genuine pairs.
Đang tạo imposter pairs...
Đã tạo 264 imposter pairs.
Tìm thấy 70 danh tính.
Đang tạo genuine pairs...
Đã tạo 247 genuine pairs.
Đang tạo imposter pairs...
Đã tạo 277 imposter pairs.


In [4]:
save_pairs_to_csv(genuine_pairs_test, "datasets/ver1/selfies_1/genuine_pairs_test.csv")
save_pairs_to_csv(
    imposter_pairs_test, "datasets/ver1/selfies_1/imposter_pairs_test.csv"
)
save_pairs_to_csv(
    genuine_pairs_train, "datasets/ver1/selfies_2/genuine_pairs_train.csv"
)
save_pairs_to_csv(
    imposter_pairs_train, "datasets/ver1/selfies_2/imposter_pairs_train.csv"
)

Đã lưu 247 pairs vào datasets/ver1/selfies_1/genuine_pairs_test.csv
Đã lưu 277 pairs vào datasets/ver1/selfies_1/imposter_pairs_test.csv
Đã lưu 239 pairs vào datasets/ver1/selfies_2/genuine_pairs_train.csv
Đã lưu 264 pairs vào datasets/ver1/selfies_2/imposter_pairs_train.csv
